# Libraries

In [2]:
from graph_creation import create_mesh_dataset, save_database, create_dataset_folders

In [3]:
create_dataset_folders(dataset_folder='datasets')

In [4]:
# Run this cell ONCE to invert the mesh scale ordering in the template (0=finest, 3=coarsest).
# After running, re-run the SFINCS workflow cell below to regenerate the datasets.
import pickle, sys
sys.path.insert(0, ".")
from graph_creation import invert_scale_ordering

template_pkl = "./datasets/train/template_marg.pkl"
with open(template_pkl, "rb") as f:
    tpl = pickle.load(f)

tpl[0] = invert_scale_ordering(tpl[0])

# Verify: meshes[0] should now have more nodes than meshes[1] (finest first)
mesh = tpl[0].mesh
sizes = [mesh.meshes[i].num_faces for i in range(mesh.num_meshes)]
print("Face counts per scale (should decrease):", sizes)

with open(template_pkl, "wb") as f:
    pickle.dump(tpl, f)
print("Template saved with inverted scale ordering.")


c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg\database\graph_creation.py:1271: RuntimeWarning: invalid value encountered in divide
  self.edge_outward_normal = self.edge_relative_distance/self.edge_length[:,None]


Face counts per scale (should decrease): [28192, 12188, 3048, 98]
Template saved with inverted scale ordering.


# Create and save pytorch geometric dataset

In [5]:
# â”€â”€ Template â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
TEMPLATE_PKL = './datasets/train/template_marg.pkl'

# â”€â”€ Simulations to convert â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Each tuple: (sfincs_dir, dataset_name, out_split)
#   sfincs_dir  : folder containing sfincs_map.nc, sfincs.src, sfincs.dis
#   dataset_name: output .pkl name (without .pkl)
#   out_split   : "train" or "test"

SFINCS_DIR = './raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart'

simulations = [
    (SFINCS_DIR, 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart', 'train'),
    (SFINCS_DIR, 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart', 'test'),
]

OUT_ROOT = './datasets'

# â”€â”€ SFINCS variable names (adjust if your map uses different names) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR          = 'u'   # set to None if not present
VY_VAR          = 'v'   # set to None if not present


In [6]:
import os, sys, copy, pickle
import numpy as np
import torch
import xarray as xr
sys.path.insert(0, ".")

from convert_sfincs_to_pkl_marg import (
    load_single_data_object,
    get_target_points,
    get_source_points,
    interpolate_time_series,
    parse_src_file,
    parse_dis_file,
    build_output_data,
)

print("Loading template...")
template_data = load_single_data_object(TEMPLATE_PKL)
target_points = get_target_points(template_data)
print("  Template mesh faces:", target_points.shape[0])

for sfincs_dir, dataset_name, out_split in simulations:
    sfincs_map = os.path.join(sfincs_dir, "sfincs_map.nc")
    src_file   = os.path.join(sfincs_dir, "sfincs.src")
    dis_file   = os.path.join(sfincs_dir, "sfincs.dis")

    print("")
    print("===", dataset_name, "->", out_split, "===")
    ds = xr.open_dataset(sfincs_map, decode_times=False)
    source_points = get_source_points(ds)

    zs = ds[WATER_LEVEL_VAR].values
    zb = ds[BED_LEVEL_VAR].values
    WD_grid = np.maximum(zs - zb[None, :, :], 0.0).astype(np.float32)
    print("  Interpolating WD...")
    WD = interpolate_time_series(source_points, WD_grid, target_points, "WD")

    ds_raw = xr.open_dataset(sfincs_map, decode_times=False, mask_and_scale=False)
    if VX_VAR and VX_VAR in ds.data_vars:
        VX_raw = ds_raw[VX_VAR].values.astype(np.float32)
        fv = ds_raw[VX_VAR].attrs.get("_FillValue", None)
        if fv is not None: VX_raw[VX_raw == fv] = np.nan
        print("  Interpolating VX...")
        VX = interpolate_time_series(source_points, VX_raw, target_points, "VX")
    else:
        VX = np.zeros_like(WD)
    if VY_VAR and VY_VAR in ds.data_vars:
        VY_raw = ds_raw[VY_VAR].values.astype(np.float32)
        fv = ds_raw[VY_VAR].attrs.get("_FillValue", None)
        if fv is not None: VY_raw[VY_raw == fv] = np.nan
        print("  Interpolating VY...")
        VY = interpolate_time_series(source_points, VY_raw, target_points, "VY")
    else:
        VY = np.zeros_like(WD)
    ds_raw.close()

    time_var = ds.coords.get("time", ds.coords.get("t", None))
    map_times_s = time_var.values.astype(np.float64) if time_var is not None else np.arange(zs.shape[0]) * 3600.0

    print("  Reading src/dis files...")
    src_xy = parse_src_file(src_file)
    dis_times_s, discharge = parse_dis_file(dis_file)
    print(" ", len(src_xy), "source points, discharge shape:", discharge.shape)

    data_out = build_output_data(
        template_data, WD=WD, VX=VX, VY=VY,
        map_times_s=map_times_s, src_xy=src_xy,
        dis_times_s=dis_times_s, discharge=discharge,
    )

    out_dir = os.path.join(OUT_ROOT, out_split)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, dataset_name + ".pkl")
    with open(out_path, "wb") as f:
        pickle.dump([data_out], f)
    print("  Saved", out_path)
    print("  WD=", tuple(data_out.WD.shape), "| node_BC=", data_out.node_BC.tolist(), "| BC=", tuple(data_out.BC.shape))


SyntaxError: unterminated string literal (detected at line 28) (2083305481.py, line 28)